In [1]:
import tdb, re
import pandas as pd
from tqdm import tqdm

In [2]:
manifest = pd.read_csv("../../adaptive_sampling_15.csv")
adotto_strchive_df = pd.read_csv("../tr_regions/strchive/adotto_strchive_intersected.bed", sep="\t", header=None, names=['chrom', 'start', 'end', 'adotto', 'strchive_chr', 'strchive_start', 'strchive_end', 'id', 'gene','motif', 'motif_alt', 'pathogenic_min', 'inheritance', 'disease', 'overlap'])

In [3]:
manifest

,run_id,sample_id,unique_id,kit_type,bed_file,proband_gender,path_to_flowcell
0,PCA100035_2025-05-20_Run1,10_12_Gregor_Trio,20250520_1236_1F_PBE50851_c0f950c7,SQK-NBD114-96,Arthrogryposis,Female,/stornext/snfs190/next-gen/ONT/PromethION/PCA1...
1,PCA100035_2025-05-20_Run1,13_15_Gregor_Trio,20250520_1237_1G_PBE41795_4209257d,SQK-NBD114-96,Microcephaly,Male,/stornext/snfs190/next-gen/ONT/PromethION/PCA1...
2,PCA100035_2025-05-20_Run1,16_18_Gregor_Trio,20250520_1237_1H_PBE51085_71daceab,SQK-NBD114-96,Microcephaly,Female,/stornext/snfs190/next-gen/ONT/PromethION/PCA1...
3,PCA100035_2025-05-20_Run1,19_21_Gregor_Trio,20250520_1237_2A_PBE56364_33e008b1,SQK-NBD114-96,Microcephaly,Female,/stornext/snfs190/next-gen/ONT/PromethION/PCA1...
4,PCA100035_2025-05-20_Run1,1_3_Gregor_Trio,20250520_1234_1A_PBE28886_15240170,SQK-NBD114-96,Epilepsy,Male,/stornext/snfs190/next-gen/ONT/PromethION/PCA1...
5,PCA100035_2025-05-20_Run1,22_24_Gregor_Trio,20250520_1238_2B_PBE57440_6d633900,SQK-NBD114-96,Neurodegeneration,Female,/stornext/snfs190/next-gen/ONT/PromethION/PCA1...
6,PCA100035_2025-05-20_Run1,4_6_Gregor_Trio,20250520_1234_1B_PBE52882_2bd2bfe2,SQK-NBD114-96,Epilepsy,Female,/stornext/snfs190/next-gen/ONT/PromethION/PCA1...
7,PCA100035_2025-05-20_Run1,7_9_Gregor_Trio,20250520_1236_1E_PBE24973_2451048f,SQK-NBD114-96,Arthrogryposis,Male,/stornext/snfs190/next-gen/ONT/PromethION/PCA1...
8,PCA100035_2025-05-20_Run2,25_27_Gregor_Trio,20250520_1247_3A_PBE56224_62f8b680,SQK-NBD114-96,Neurodegeneration,Male,/stornext/snfs190/next-gen/ONT/PromethION/PCA1...
9,PCA100035_2025-05-20_Run2,28-30_Gregor_Trio,20250520_1247_3B_PBE57257_5456e4c7,SQK-NBD114-96,Neurodegeneration,Male,/stornext/snfs190/next-gen/ONT/PromethION/PCA1...


In [4]:
def extract_barcodes(sample_id):
    if '-' in str(sample_id):
        # Range like 43-45
        start, end = map(int, re.match(r"(\d+)-(\d+)", sample_id).groups())
        return  [f"barcode{num:02d}" for num in range(start, end + 1)] 
    else:
        start, end = map(int, re.match(r"(\d+)_(\d+)", sample_id).groups())
        return [f"barcode{num:02d}" for num in range(start, end + 1)] 

In [5]:
from collections import defaultdict
from tqdm import tqdm
import pandas as pd

def find_unexpected_child_alleles(manifest, adotto_strchive_df, wiggle=0.10):
    """
    Intersect first, then compute trio logic (faster).
    """
    def is_within_wiggle(child_len, parent_lens, rel_tol=wiggle):
        # Accept if child length is within ±max(1bp, round(p*rel_tol)) of ANY parent allele length
        for p in parent_lens:
            tol = max(1, int(round(p * rel_tol)))
            if abs(int(child_len) - int(p)) <= tol:
                return True
        return False

    all_results = []

    for row in tqdm(manifest.itertuples(index=False), total=len(manifest)):
        sample_name = getattr(row, "sample_id", None)
        if not sample_name:
            continue

        try:
            tdb_entries = tdb.load_tdb(f"../trio_analysis/tr_trio/{sample_name}.tdb")
            if 'sample' not in tdb_entries or 'allele' not in tdb_entries or 'locus' not in tdb_entries:
                continue

            # --- 1) Intersect FIRST: locus ↔ Adotto/STRchive -------------------
            # Expect columns: ['LocusID','chrom','start','end'] in locus
            if not {'LocusID','chrom','start','end'}.issubset(tdb_entries['locus'].columns):
                # Can't intersect early without coordinates; fall back to computing everything
                locus_interested = None
            else:
                locus_coords = tdb_entries['locus'][['LocusID','chrom','start','end']].copy()

                # Inner join to keep only loci present in Adotto/STRchive
                # Adotto/STRchive df must have ['chrom','start','end']
                if not {'chrom','start','end'}.issubset(adotto_strchive_df.columns):
                    locus_interested = None
                else:
                    locus_hit = locus_coords.merge(
                        adotto_strchive_df[['chrom','start','end']],
                        on=['chrom','start','end'],
                        how='inner'
                    ).drop_duplicates(subset=['LocusID'])
                    # Set of LocusID to keep
                    locus_interested = set(locus_hit['LocusID'].tolist())

            # --- 2) Build [LocusID, role, allele_length] quickly ----------------
            # Build per-read/sample allele table: [sample, LocusID, allele_number]
            parts = []
            for per_sample_name, table in tdb_entries['sample'].items():
                if {'LocusID', 'allele_number'}.issubset(table.columns):
                    tmp = table[['LocusID', 'allele_number']].copy()
                    tmp['sample'] = per_sample_name
                    parts.append(tmp)
            if not parts:
                continue
            sample_locusid_allele_number = pd.concat(parts, ignore_index=True)

            # Join allele attributes (need allele_length)
            if not {'LocusID','allele_number'}.issubset(tdb_entries['allele'].columns):
                continue
            allele_cols = ['LocusID', 'allele_number']
            if 'allele_length' in tdb_entries['allele'].columns:
                allele_cols.append('allele_length')
            allele_df = tdb_entries['allele'][allele_cols].copy()

            df = (
                sample_locusid_allele_number
                .merge(allele_df, on=['LocusID','allele_number'], how='left')
            )

            # Map sample -> role (compute barcodes once)
            barcodes = extract_barcodes(sample_name)  # (child, mother, father) per your helper
            def infer_role(sample_label):
                if barcodes[0] in sample_label:
                    return 'child'
                elif barcodes[1] in sample_label:
                    return 'mother'
                elif barcodes[2] in sample_label:
                    return 'father'
                return None

            df['role'] = df['sample'].map(infer_role)
            df = df.dropna(subset=['role', 'allele_length', 'LocusID'])

            # If we have an early intersection, filter to those LocusIDs now (big speedup)
            if locus_interested is not None:
                df = df[df['LocusID'].isin(locus_interested)]
                if df.empty:
                    continue

            # Keep only what's needed downstream
            df = df[['LocusID', 'role', 'allele_length']].copy()

            # --- 3) Build per-role allele lists efficiently --------------------
            # role_lists: index=LocusID, columns=['child','mother','father'], values=list of lengths
            role_lists = (
                df.groupby(['LocusID','role'])['allele_length']
                  .apply(lambda s: s.astype(int).tolist())
                  .unstack(fill_value=[])
                  .reset_index()
            )
            # Ensure missing roles are present as empty lists
            for r in ('child','mother','father'):
                if r not in role_lists.columns:
                    role_lists[r] = [[] for _ in range(len(role_lists))]

            # --- 4) Compute unexpected child alleles per LocusID ---------------
            def compute_not_inherited(row_):
                father_alleles = row_['father']
                mother_alleles = row_['mother']
                child_alleles  = row_['child']
                not_inherited = [
                    c for c in child_alleles
                    if not (is_within_wiggle(c, father_alleles) or is_within_wiggle(c, mother_alleles))
                ]
                return pd.Series({
                    'father_alleles': father_alleles,
                    'mother_alleles': mother_alleles,
                    'child_alleles': child_alleles,
                    'not_inherited_alleles': not_inherited,
                    'n_not_inherited': len(not_inherited)
                })

            results = role_lists.join(role_lists.apply(compute_not_inherited, axis=1))
            candidates = results[results['n_not_inherited'] > 0].copy()
            if candidates.empty:
                continue

            # --- 5) Attach coordinates and intersect (already filtered, but add coords) ----
            if {'LocusID','chrom','start','end'}.issubset(tdb_entries['locus'].columns):
                coords = tdb_entries['locus'][['LocusID','chrom','start','end']].copy()
                candidates = candidates.merge(coords, on='LocusID', how='left')

                # Optional: strict join to adotto/strchive to bring in extra annotation columns
                if {'chrom','start','end'}.issubset(adotto_strchive_df.columns):
                    merged = candidates.merge(
                        adotto_strchive_df,
                        on=['chrom','start','end'],
                        how='inner' if locus_interested is not None else 'left'
                    )
                else:
                    merged = candidates
            else:
                merged = candidates

            # Add sample indicator and wiggle used
            merged['sample_name'] = sample_name
            merged['wiggle'] = wiggle
            merged['category'] = getattr(row, "bed_file", None)
            # Optional: explode to one row per unexpected child allele
            if 'not_inherited_alleles' in merged.columns:
                merged = merged.explode('not_inherited_alleles', ignore_index=True)

            all_results.append(merged)

        except Exception as e:
            all_results.append(pd.DataFrame({
                'sample_name': [sample_name],
                'error': [str(e)]
            }))

    if not all_results:
        return pd.DataFrame()
    return pd.concat(all_results, ignore_index=True)


In [6]:
final_df = find_unexpected_child_alleles(manifest, adotto_strchive_df, wiggle=0.10)

100%|██████████| 18/18 [00:07<00:00,  2.33it/s]


In [8]:
final_df[final_df['gene']=='NAXE']

,LocusID,child,father,mother,father_alleles,mother_alleles,child_alleles,not_inherited_alleles,n_not_inherited,chrom,...,gene,motif,motif_alt,pathogenic_min,inheritance,disease,overlap,sample_name,wiggle,category
6,1263,[151],[],[],[],[],[151],151,1,chr1,...,NAXE,GGGCC,GGGCC,200.0,AR,NAXE-related mitochondrial encephalopathy,18,1_3_Gregor_Trio,0.1,Epilepsy
15,1519,"[1251, 622]",[152],"[567, 152]",[152],"[567, 152]","[1251, 622]",1251,1,chr1,...,NAXE,GGGCC,GGGCC,200.0,AR,NAXE-related mitochondrial encephalopathy,18,34-36_Gregor_Trio,0.1,Neurodegeneration


In [159]:
final_df.to_csv("../trio_analysis/tr_trio_annotation/tr_trio_annotation.tsv", sep="\t")

In [9]:
from tqdm import tqdm
import pandas as pd

def collect_child_alleles_with_annotations(manifest, adotto_strchive_df, explode=True):
    """
    For each trio in `manifest`, report the CHILD's allele lengths at loci that intersect Adotto/STRchive.
    No parent checking. Returns one combined DataFrame with merged annotations.

    Args:
        manifest (pd.DataFrame): must have a `sample_id` column; `bed_file` is optional (stored as `category`).
        adotto_strchive_df (pd.DataFrame): must include ['chrom','start','end'] plus any annotation columns.
        explode (bool): if True, return one row per child allele (exploded). If False, keep list per locus.

    Returns:
        pd.DataFrame
            Columns include:
              - sample_name, category, LocusID, chrom, start, end
              - child_alleles (list) and n_child_alleles
              - all columns from `adotto_strchive_df` (merged on chrom/start/end)
    """
    required_adotto_cols = {'chrom', 'start', 'end'}
    if not required_adotto_cols.issubset(adotto_strchive_df.columns):
        raise ValueError(f"adotto_strchive_df must contain columns {sorted(required_adotto_cols)}")

    # De-duplicate adotto/strchive rows on the interval keys to avoid large cartesian merges later
    adotto_keydedup = adotto_strchive_df.drop_duplicates(subset=['chrom', 'start', 'end']).copy()

    all_results = []
    for row in tqdm(manifest.itertuples(index=False), total=len(manifest)):
        sample_name = getattr(row, "sample_id", None)
        if not sample_name:
            continue

        try:
            tdb_entries = tdb.load_tdb(f"../trio_analysis/tr_trio/{sample_name}.tdb")
            if not {'sample','allele','locus'}.issubset(set(tdb_entries.keys())):
                continue

            # --- Intersect FIRST: locus ↔ Adotto/STRchive -------------------
            if not {'LocusID','chrom','start','end'}.issubset(tdb_entries['locus'].columns):
                # Without coordinates, we can't intersect; skip
                continue

            locus_coords = tdb_entries['locus'][['LocusID','chrom','start','end']].copy()
            locus_hit = (
                locus_coords
                .merge(adotto_keydedup[['chrom','start','end']], on=['chrom','start','end'], how='inner')
                .drop_duplicates(subset=['LocusID'])
            )
            if locus_hit.empty:
                continue
            locus_interested = set(locus_hit['LocusID'])

            # --- Build per-read/sample allele table and keep CHILD only ------
            # Concatenate per-sample tables: [sample, LocusID, allele_number]
            parts = []
            for per_sample_name, table in tdb_entries['sample'].items():
                if {'LocusID', 'allele_number'}.issubset(table.columns):
                    tmp = table[['LocusID', 'allele_number']].copy()
                    tmp['sample'] = per_sample_name
                    parts.append(tmp)
            if not parts:
                continue
            per_read = pd.concat(parts, ignore_index=True)

            # Map sample -> role; retain only child
            barcodes = extract_barcodes(sample_name)  # expected order: (child, mother, father)
            child_tag = barcodes[0]

            # Fast boolean mask for child rows
            child_rows = per_read['sample'].astype(str).str.contains(child_tag, na=False)
            per_read_child = per_read.loc[child_rows]
            if per_read_child.empty:
                continue

            # Filter to intersected loci
            per_read_child = per_read_child[per_read_child['LocusID'].isin(locus_interested)]
            if per_read_child.empty:
                continue

            # Join allele attributes to get allele_length
            if not {'LocusID','allele_number'}.issubset(tdb_entries['allele'].columns):
                continue
            allele_cols = ['LocusID', 'allele_number']
            if 'allele_length' in tdb_entries['allele'].columns:
                allele_cols.append('allele_length')
            allele_df = tdb_entries['allele'][allele_cols].copy()

            df = per_read_child.merge(allele_df, on=['LocusID','allele_number'], how='left')
            df = df.dropna(subset=['allele_length'])

            # Keep only what we need
            df = df[['LocusID', 'allele_length']].copy()

            # Aggregate child alleles per LocusID
            child_by_locus = (
                df.groupby('LocusID')['allele_length']
                  .apply(lambda s: s.astype(int).tolist())
                  .reset_index(name='child_alleles')
            )
            child_by_locus['n_child_alleles'] = child_by_locus['child_alleles'].map(len)

            # Attach coordinates
            child_by_locus = child_by_locus.merge(
                locus_coords, on='LocusID', how='left'
            )

            # Attach all Adotto/STRchive annotations
            merged = child_by_locus.merge(
                adotto_keydedup, on=['chrom','start','end'], how='inner'
            )

            # Add indicators
            merged['sample_name'] = sample_name
            merged['category'] = getattr(row, "bed_file", None)

            # Optional: explode to one row per child allele
            if explode:
                merged = merged.explode('child_alleles', ignore_index=True)
                merged = merged.rename(columns={'child_alleles': 'child_allele'})
                # Preserve n_child_alleles as locus-level count

            all_results.append(merged)

        except Exception as e:
            all_results.append(pd.DataFrame({
                'sample_name': [sample_name],
                'error': [str(e)]
            }))

    if not all_results:
        return pd.DataFrame()
    return pd.concat(all_results, ignore_index=True)


In [10]:
final_df = collect_child_alleles_with_annotations(manifest, adotto_strchive_df)

100%|██████████| 18/18 [00:03<00:00,  5.43it/s]


In [11]:
df = final_df
df['child_allele'] = df['child_allele'].astype(int)
df['pathogenic_min'] = pd.to_numeric(df['pathogenic_min'], errors='coerce')

# Compute repeat size in base pairs: allele length × motif length
df['allele_bp'] = df.apply(lambda r: r['child_allele'] / len(str(r['motif'])), axis=1)

# Filter by threshold
df_filtered = df[df['allele_bp'] > df['pathogenic_min']].copy()
df_filtered = df_filtered.drop(columns=['allele_bp'])


In [167]:
df_filtered.to_csv("../trio_analysis/tr_trio_annotation/tr_trio_annotation_child_only.tsv", sep="\t")

In [12]:
df_filtered

,LocusID,child_allele,n_child_alleles,chrom,start,end,adotto,strchive_chr,strchive_start,strchive_end,id,gene,motif,motif_alt,pathogenic_min,inheritance,disease,overlap,sample_name,category
8,13378,309,2,chr9,130681497,130681800,ID=1709759;MOTIFS=CGCGCACGCGCCCGCGCTGCACCGCCCC...,chr9,130681605,130681641,HSAN-VIII_PRDM12,PRDM12,GCC,GCC,18.0,AR,Hereditary sensory and autonomic neuropathy ty...,36,10_12_Gregor_Trio,Arthrogryposis
9,13378,303,2,chr9,130681497,130681800,ID=1709759;MOTIFS=CGCGCACGCGCCCGCGCTGCACCGCCCC...,chr9,130681605,130681641,HSAN-VIII_PRDM12,PRDM12,GCC,GCC,18.0,AR,Hereditary sensory and autonomic neuropathy ty...,36,10_12_Gregor_Trio,Arthrogryposis
16,17564,109,2,chr14,23321433,23321542,ID=458324;MOTIFS=GCG;STRUC=(GCG)n,chr14,23321472,23321503,OPMD_PABPN1,PABPN1,GCN,GCN,12.0,"AD,AR",Oculopharyngeal muscular dystrophy,31,10_12_Gregor_Trio,Arthrogryposis
17,17564,109,2,chr14,23321433,23321542,ID=458324;MOTIFS=GCG;STRUC=(GCG)n,chr14,23321472,23321503,OPMD_PABPN1,PABPN1,GCN,GCN,12.0,"AD,AR",Oculopharyngeal muscular dystrophy,31,10_12_Gregor_Trio,Arthrogryposis
20,19832,179,2,chr16,67842832,67843011,ID=602292;MOTIFS=GCA;STRUC=(GCA)n,chr16,67842862,67842950,SCA_THAP11,THAP11,CAG,CAG,45.0,AD,Spinocerebellar ataxia,88,10_12_Gregor_Trio,Arthrogryposis
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
761,329282,164,1,chrX,140504285,140504449,ID=1769313;MOTIFS=GCGGCA;STRUC=(GCGGCA)n,chrX,140504316,140504361,XLMR_SOX3,SOX3,NGC,NGC,22.0,XR,X-linked panhypopituitarism ; X-linked mental ...,45,1_3_wgs_HG002_Trio,WGS
768,2599,163,2,chr19,45770173,45770363,ID=756441;MOTIFS=CAG;STRUC=(CAG)n,chr19,45770204,45770266,DM1_DMPK,DMPK,CAG,CAG,50.0,AD,Myotonic dystrophy type 1,62,4_6_HG002_Trio,CMRG
769,2599,163,2,chr19,45770173,45770363,ID=756441;MOTIFS=CAG;STRUC=(CAG)n,chr19,45770204,45770266,DM1_DMPK,DMPK,CAG,CAG,50.0,AD,Myotonic dystrophy type 1,62,4_6_HG002_Trio,CMRG
774,2729,163,2,chr19,45770173,45770363,ID=756441;MOTIFS=CAG;STRUC=(CAG)n,chr19,45770204,45770266,DM1_DMPK,DMPK,CAG,CAG,50.0,AD,Myotonic dystrophy type 1,62,1_3_HG002_Trio,CMRG
